In [40]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:10pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:80px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:2px;}
table.dataframe{font-size:10pt;} 
</style>
"""))

<font size="6" color="green"><b>ch14. 웹 데이터 수집</b></font>
# 1절.Selenium을 이용한 동적 웹크롤링 문법

- https://selenium-python.readthedocs.io 

- pip install selenium(아나콘다 프롬프트) 
       - 경고 무지 => pip install --upgrade requests (requests를 최신버전으로 upgrade)
                     conda install urllib3 == 1.26.18

<br>
selenium버전 : 4.47.0 / requests버전 : 2.28.1 / urllib버전 : 2.7.0

In [2]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
import time

In [3]:
dv = webdriver.Chrome()
dv.get('http://python.org')

In [12]:
elem = dv.find_element(By.NAME, 'q')
# By.CLASS_NAME, By.ID,By.CSS_SELECTOR, By.TAG_NAME
# a태그에서 By.LINK_TEXT, by.PARTIAL_LINK_TEXT
elem.clear()
elem.send_keys('pycon')
elem.send_keys(Keys.RETURN) # enter

In [15]:
elem = dv.find_element(By.NAME, 'q')
elem.send_keys(Keys.CONTROL, 'a') # ctrl + a
elem.send_keys('pycon')
btn_elem = dv.find_element(By.CSS_SELECTOR, 'button#submit') # Go버튼
btn_elem.click()

In [18]:
result_list = dv.find_elements(By.CSS_SELECTOR, 'li>h3>a')
# len(result_list)
for result in result_list:
    print("{} - {}".format(result.text, result.get_attribute('href')))

PSF PyCon Trademark Usage Policy - https://www.python.org/psf/trademarks/pycon
PyCon Italia 2016 (PyCon Sette) - https://www.python.org/events/python-events/378/
PyCon Australia 2013 - https://www.python.org/events/python-events/57/
PyCon AU 2019 - https://www.python.org/events/python-events/776/
PyCon NL 2025 - https://www.python.org/events/python-events/2084/
PyCon Australia 2014 - https://www.python.org/events/python-events/10/
PyCon Ireland 2012 - https://www.python.org/events/python-events/76/
PyCon Ireland 2016 - https://www.python.org/events/python-events/429/
PyCon Ireland 2022 - https://www.python.org/events/python-events/1320/
PyCon Australia 2014 - https://www.python.org/events/python-events/1447/
PyCon Ireland 2023 - https://www.python.org/events/python-events/1568/
PyCon Ireland 2024 - https://www.python.org/events/python-events/1862/
PyCon APAC 2025 - https://www.python.org/events/python-events/1879/
PyCon AU 2018 - https://www.python.org/events/python-events/696/
PyCon A

In [21]:
from bs4 import BeautifulSoup
soup = BeautifulSoup(dv.page_source, 'html.parser')
result_list = soup.select('li>h3>a')
for result in result_list[:3]:
    print("{} - {}".format(result.text, result.attrs.get('href')))

PSF PyCon Trademark Usage Policy - /psf/trademarks/pycon
PyCon Italia 2016 (PyCon Sette) - /events/python-events/378/
PyCon Australia 2013 - /events/python-events/57/


In [23]:
from urllib.parse import urlparse
current_url = dv.current_url
print('현재 url :', current_url)
result_parse = urlparse(current_url)
print('url parsing 결과 :', result_parse)
domain = f'{result_parse.scheme}://{result_parse.netloc}'
#== domain = "{}://{}".format(result_parse.scheme, result_parse.netloc)
print('현재 domain :', domain)

현재 url : https://www.python.org/search/?q=pycon&submit=
url parsing 결과 : ParseResult(scheme='https', netloc='www.python.org', path='/search/', params='', query='q=pycon&submit=', fragment='')
현재 domain : https://www.python.org


In [24]:
soup = BeautifulSoup(dv.page_source, 'html.parser')
result_list = soup.select('li>h3>a')
for result in result_list[:3]:
    print("{} - {}".format(result.text, domain+result.attrs.get('href')))

PSF PyCon Trademark Usage Policy - https://www.python.org/psf/trademarks/pycon
PyCon Italia 2016 (PyCon Sette) - https://www.python.org/events/python-events/378/
PyCon Australia 2013 - https://www.python.org/events/python-events/57/


In [25]:
dv.close() # 브라우저 종료하기

# 2절. 동적웹크롤링 예제
## 2.1 다음 뉴스 검색

In [26]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time
news_list = [] # 뉴스제목과 뉴스 link들을 저장할 list
driver = webdriver.Chrome()
url = 'https://www.daum.net/'
driver.get(url)
time.sleep(0.5) # 다음페이지가 다 뜰 때까지 0.5초 대기

query = input('검색할 단어는?')
driver.find_element(By.CLASS_NAME, 'tf_keyword').send_keys(query)
driver.find_element(By.CSS_SELECTOR, 'button[type=submit]').click()
time.sleep(2) # 페이지 로딩될 시간동안 대기하기
# 뉴스 탭 클릭
# driver.find_elements(By.CSS_SELECTOR, 'ul.list_tab > li')[1].click()
driver.find_element(By.LINK_TEXT, '뉴스').click()

검색할 단어는?약과


In [27]:
bodies = driver.find_elements(By.CSS_SELECTOR, 'strong.tit-g.clamp-g')
# len(bodies)
for body in bodies:
    a = body.find_element(By.TAG_NAME, 'a')
    title = a.text
    link  = a.get_attribute('href')
    # print(title, link)
    news_list.append([title, link])

In [28]:
page_nav = driver.find_element(By.CLASS_NAME, 'inner_paging')
# page_nav.text
nex_page = page_nav.find_element(By.LINK_TEXT, "4") # a태그의 text가 2인 a태그
nex_page.click()

In [29]:
import pandas as pd
pd.DataFrame(news_list, columns=['뉴스제목','링크']).shape

(10, 2)

## 2-2 다음뉴스 페이징 처리
위의 예제를 이용하여 원하는 페이지만큼 뉴스 검색 결과를 받아오기

In [30]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time
news_list = [] # 뉴스제목과 뉴스 link들을 저장할 list
driver = webdriver.Chrome()
url = 'https://www.daum.net/'
driver.get(url)
time.sleep(0.5) # 다음페이지가 다 뜰 때까지 0.5초 대기

# query = 'AI'
query = input('검색할 단어는?')
driver.find_element(By.CLASS_NAME, 'tf_keyword').send_keys(query)
driver.find_element(By.CSS_SELECTOR, 'button[type=submit]').click()
time.sleep(2) # 페이지 로딩될 시간동안 대기하기
# 뉴스 탭 클릭
# driver.find_elements(By.CSS_SELECTOR, 'ul.list_tab > li')[1].click()
driver.find_element(By.LINK_TEXT, '뉴스').click()

pages = int(input('몇 페이지 크롤링 할까요?'))
for page in range(1, pages+1):
    bodies = driver.find_elements(By.CSS_SELECTOR, 'strong.tit-g.clamp-g')
    for body in bodies:
        a = body.find_element(By.TAG_NAME, 'a')
        title = a.text
        link  = a.get_attribute('href')
        # print(title, link)
        news_list.append([title, link])
    page_nav = driver.find_element(By.CLASS_NAME, 'inner_paging')
    nex_page = page_nav.find_element(By.LINK_TEXT, str(page+1)) # a태그의 text가 2인 a태그
    nex_page.click()
    time.sleep(2)
driver.close()
news_df = pd.DataFrame(news_list, columns=['title','link'])
display(news_df.head())
print(news_df.shape)

검색할 단어는?추풍낙엽
몇 페이지 크롤링 할까요?5


,title,link
0,"""'매국노대추풍낙엽차' 시키면 '이완용 얼굴' 드려요""…광복절 '매국노 조롱잔치' ...",http://v.daum.net/v/20260812141242283
1,"“이의리가 마무리도 할 수 있나요” 네, 합니다…전율의 첫 SV, 156km로 구자...",http://v.daum.net/v/20260811230150815
2,'추풍낙엽' 허위사실공표죄…이대로 괜찮나 [이브닝 브리핑],http://v.daum.net/v/20260728180018370
3,24살 '트랙 여신' 혼자 金 4개 쓸어담았다…독주 2관왕 이어 계주에서도 9초86...,http://v.daum.net/v/20260817140637288
4,‘매국노 조롱 카페’ 오픈런하고 ‘조상 중 친일파 몇명’ 찾아보고,http://v.daum.net/v/20260817004254977


(50, 2)


## 맞춤법 검사기
네이버 맞춤법 검사기 이용

In [31]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup
import time

In [32]:
driver = webdriver.Chrome()
driver.get('https://www.naver.com')
time.sleep(1)
elem = driver.find_element(By.ID, 'query')
elem.send_keys(Keys.CONTROL, 'a') # input이나 textarea나 다른 태그
elem.send_keys('맞춤법 검사기')
elem.send_keys(Keys.RETURN)
time.sleep(1)
textarea = driver.find_element(By.CLASS_NAME, 'txt_gray')
textarea.clear() # input이나 textarea
textarea.send_keys('안뇽하세요. 방갑습니다. 맛있는 점심시간 되세용')
btn = driver.find_element(By.CLASS_NAME, 'btn_check')
btn.click()
time.sleep(2)
result = driver.find_element(By.CSS_SELECTOR, 'p._result_text.stand_txt').text
print(result)
driver.close()

안녕하세요. 반갑습니다. 맛있는 점심시간 되세요


### 맞춤법검사전.txt파일을 맞춤법검사후.txt로 파일 출력

In [39]:
# fp = open('data/ch14_맞춤법검사_전.txt', 'r', encoding='utf-8')
# text = fp.read()
# fp.close()
with open('Data/ch14_맞춤법검사_전.txt', 'r', encoding='utf-8') as fp:
    text = fp.read()
ready_text_list = [] # 300자 기준으로 문장단위로 나눠진 text list
while len(text)>=300:
    temp = text[:300]
    last_dot_index = temp.rfind('.')
    ready_text_list.append(text[:last_dot_index+1])
    text = text[last_dot_index+1:]
ready_text_list.append(text)
print([len(ready_text) for ready_text in ready_text_list])

[282, 249, 198]


In [44]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup
import time
driver = webdriver.Chrome()
driver.get('https://www.naver.com')
time.sleep(1)
elem = driver.find_element(By.ID, 'query')

elem.send_keys('맞춤법 검사기')
elem.send_keys(Keys.RETURN)
time.sleep(0.5)
textarea = driver.find_element(By.CLASS_NAME, 'txt_gray')
results = '' # 맞춤법 검사 완료된 text
for idx, ready_text in enumerate(ready_text_list):
    print(f'검사중...{idx+1}/{len(ready_text_list)}')
    textarea.clear() # input이나 textarea
    textarea.send_keys(ready_text)
    btn = driver.find_element(By.CLASS_NAME, 'btn_check')
    btn.click()
    time.sleep(1)
    # result = driver.find_element(By.CSS_SELECTOR, 'p._result_text.stand_txt').text
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    result = soup.select_one('p._result_text.stand_txt').text
    results += result + ' '
    result = results.replace('.', '. ')
    
driver.close()

검사중...1/3
검사중...2/3
검사중...3/3


In [45]:
# 맞춤법 검사 결과(result)를 파일 출력
with open('Data/ch14_맞춤법검사_후.txt', 'w') as fp:
    fp.write(results)

# 3절 연습문제.
- https://papago.naver.com/ 을 통해서 "data/ch14_맞춤법후.txt"파일의 내용을 영문으로 번역하여 파일 출력

In [47]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup
import time

In [59]:
with open('Data/ch14_맞춤법검사_후.txt', 'r') as fp:
    text = fp.read()
ready_text_list = [] # 300자 기준으로 문장단위로 나눠진 text list
while len(text)>=3000:
    temp = text[:3000]
    last_dot_index = temp.rfind('.')
    ready_text_list.append(text[:last_dot_index+1])
    text = text[last_dot_index+1:]
ready_text_list.append(text)
print([len(ready_text) for ready_text in ready_text_list])

[718]


In [60]:
driver = webdriver.Chrome()
driver.get('https://papago.naver.com/')

time.sleep(0.5)
btn = driver.find_element(By.CLASS_NAME, 'entry-popup-module-scss-module__UZIxta__close')
if btn:
    btn.click()
    print('축하 창 닫음 ')
else:
    print('축하창 안 뜸')
input_elem = driver.find_element(By.CLASS_NAME,
                                 'text-translator-module-scss-module__CYJRkW__text-editor')
results = '' #번역결과를 담을 변수
for i, ready_text in enumerate(ready_text_list):
    input_elem.send_keys(Keys.CONTROL, 'a')
    input_elem.send_keys(ready_text)
    time.sleep(2)
    result = driver.find_elements(By.CLASS_NAME,
                             'text-editor-module-scss-module__gKzuvW__dynamic-smd')[1].text
    results += result +''
driver.close()
with open('Data/ch14_자동화영어번역본.txt', 'w') as fp:
    fp.write(results)

축하 창 닫음 
